# Case study parsing — run everything here (Colab GPU)

**If your birth-certificate images live in this repo** (e.g. `data/raw_images/DataSet/Birth Certificate/`), you **do not need Google Drive**. Push the repo to GitHub **including `data/`**, then use **`SOURCE = "github"`** — Colab will `git clone` and the images come with it.

**Pick one:**

- **`github`** (recommended when images are in git) — push your repo, set `REPO_URL`, clone on Colab.
- **`drive`** — only if the full project folder exists only on Google Drive, not on GitHub.

**Colab:** [colab.research.google.com](https://colab.research.google.com) → upload **this notebook** → **Runtime → Change runtime type → GPU** → run top to bottom.

**GitHub limits:** any single file **> 100 MB** cannot be pushed unless you use **Git LFS**. Huge folders may need Drive + optional copy step instead.

In [37]:
import torch
print("torch:", torch.__version__)
print("cuda available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("device:", torch.cuda.get_device_name(0))

torch: 2.10.0+cu128
cuda available: True
device: Tesla T4


## 1) Configure — edit this cell

**Images inside the repo:** keep **`SOURCE = "github"`**, then:

1. Push this project to GitHub (including `data/raw_images/...`).
2. On GitHub open your repo → green **Code** → **HTTPS** → copy the URL (looks like `https://github.com/you/repo.git`).
3. In the next cell set **`REPO_URL = "https://github.com/..."`** (same string, in quotes).

**Only use `SOURCE = "drive"`** if you never push to GitHub and keep the project copy only on Google Drive.

In [38]:
# --- edit below ---
SOURCE = "github"  # "github" = clone repo (images in repo) | "drive" = project only on Google Drive

# If SOURCE == "github": REQUIRED — paste your repo URL from GitHub → Code → HTTPS (must not stay empty).
REPO_URL = "https://github.com/iMalakAhmed/Case-Study-Parsing-.git"
CLONE_DIR = "/content/Case-Study-Parsing-"

# If SOURCE == "drive": path to the repo root (must contain case_study_parser/, data/, ...)
DRIVE_PROJECT_PATH = "/content/drive/MyDrive/GRAD/AI/Case-Study-Parsing-"

# If you cloned from GitHub but BC images only exist on Drive, set these after mounting Drive:
COPY_BC_FROM_DRIVE = False
DRIVE_BC_FOLDER = "/content/drive/MyDrive/path/to/Birth Certificate"
REPO_BC_DEST = "data/raw_images/DataSet/Birth Certificate"
# --- stop editing ---

## 2) Open the project (clone **or** Drive)

Clones from GitHub **or** opens your folder on Drive. Birth-certificate images are **found automatically** in step 6 (canonical paths + `BC_*.jpeg` anywhere under the repo).

In [39]:
import os
import re
import shutil
import subprocess
import sys
import tempfile
import urllib.error
import urllib.request
import zipfile


def parse_github_owner_repo(https_url: str):
    u = https_url.strip().rstrip("/")
    if u.endswith(".git"):
        u = u[:-4]
    m = re.match(r"https?://github\.com/([^/]+)/([^/]+)/?$", u)
    if not m:
        return None
    return m.group(1), m.group(2)


def clone_or_download_github(url: str, dest: str) -> None:
    # If the kernel's cwd was inside `dest`, deleting dest breaks git subprocess:
    # fatal: Unable to read current working directory
    safe_base = "/content" if os.path.isdir("/content") else "/tmp"
    os.chdir(safe_base)

    if os.path.lexists(dest):
        shutil.rmtree(dest, ignore_errors=True)

    stderr_all: list[str] = []
    for cmd in (
        ["git", "clone", "--depth", "1", url, dest],
        ["git", "clone", url, dest],
    ):
        print("$", " ".join(cmd))
        proc = subprocess.run(cmd, capture_output=True, text=True, cwd=safe_base)
        if proc.stdout:
            print(proc.stdout, end="")
        if proc.stderr:
            print(proc.stderr, end="", file=sys.stderr)
            stderr_all.append(f"{' '.join(cmd)} stderr:\n{proc.stderr}")
        if proc.returncode == 0:
            return

    parsed = parse_github_owner_repo(url)
    if parsed:
        owner, repo = parsed
        print(f"Trying ZIP download (public repos only): {owner}/{repo} …")
        zip_path = os.path.join(tempfile.gettempdir(), "github_repo_dl.zip")
        unpack_root = os.path.join(tempfile.gettempdir(), "github_zip_root")
        shutil.rmtree(unpack_root, ignore_errors=True)
        for branch in ("main", "master"):
            zip_url = f"https://github.com/{owner}/{repo}/archive/refs/heads/{branch}.zip"
            try:
                urllib.request.urlretrieve(zip_url, zip_path)
                if os.path.getsize(zip_path) < 500:
                    stderr_all.append(f"ZIP {branch}: tiny file (wrong branch?)")
                    continue
                with zipfile.ZipFile(zip_path, "r") as zf:
                    top = zf.namelist()[0].split("/")[0]
                    zf.extractall(unpack_root)
                inner = os.path.join(unpack_root, top)
                shutil.move(inner, dest)
                print("ZIP fallback OK.")
                return
            except (urllib.error.HTTPError, OSError, zipfile.BadZipFile) as exc:
                stderr_all.append(f"ZIP {branch}: {exc}")
                continue
            finally:
                shutil.rmtree(unpack_root, ignore_errors=True)

    raise RuntimeError(
        "Could not get the repo. Full diagnostics:\n\n"
        + "\n\n".join(stderr_all)
        + f"\n\nFixes: (1) Private repo: REPO_URL = https://TOKEN@github.com/owner/repo.git "
        f"(2) Run !rm -rf {dest!r} then retry (3) Confirm URL in browser."
    )


if SOURCE == "github":
    url = REPO_URL.strip()
    if not url.startswith(("https://", "http://")):
        raise ValueError(
            "REPO_URL is empty or not an http(s) URL. On GitHub: Code → HTTPS → copy → paste into "
            'REPO_URL in the cell above. Push your repo first. Or set SOURCE = "drive" if you only use Google Drive.'
        )
    clone_or_download_github(url, CLONE_DIR)
    os.chdir(CLONE_DIR)
elif SOURCE == "drive":
    from google.colab import drive

    drive.mount("/content/drive")
    if not os.path.isdir(DRIVE_PROJECT_PATH):
        raise FileNotFoundError(DRIVE_PROJECT_PATH)
    os.chdir(DRIVE_PROJECT_PATH)
else:
    raise ValueError('SOURCE must be "github" or "drive"')

print("cwd:", os.getcwd())


$ git clone --depth 1 https://github.com/iMalakAhmed/Case-Study-Parsing-.git /content/Case-Study-Parsing-
cwd: /content/Case-Study-Parsing-


Cloning into '/content/Case-Study-Parsing-'...


## 3) Optional — copy birth certificates from Drive (GitHub clone only)

Skip if `data/raw_images/DataSet/Birth Certificate/` is already in the repo.

Set **`COPY_BC_FROM_DRIVE = True`** in the config cell and fix **`DRIVE_BC_FOLDER`**. This cell mounts Drive if needed.

In [40]:
import os
import shutil

if not COPY_BC_FROM_DRIVE:
    print("Skipping (COPY_BC_FROM_DRIVE is False).")
else:
    from google.colab import drive
    drive.mount("/content/drive")
    dest = os.path.join(os.getcwd(), REPO_BC_DEST)
    os.makedirs(dest, exist_ok=True)
    for name in os.listdir(DRIVE_BC_FOLDER):
        src = os.path.join(DRIVE_BC_FOLDER, name)
        if os.path.isfile(src):
            shutil.copy2(src, os.path.join(dest, name))
    print("Copied BC images to:", dest)

Skipping (COPY_BC_FROM_DRIVE is False).


## 4) Install dependencies

Run **section 2** first. This cell looks for `requirements.txt` in the clone, **downloads it from GitHub raw** if missing, or installs a **built-in fallback** list (same as your local `requirements.txt`) — then **`git push requirements.txt`** when you can.

In [42]:
import os
import re
import subprocess
import sys
import urllib.request
from pathlib import Path


def find_requirements_file() -> tuple[str, str]:
    """Return (directory_to_chdir, path_to_requirements_txt)."""
    cwd_req = Path(os.getcwd()) / "requirements.txt"
    if cwd_req.is_file():
        return str(os.getcwd()), str(cwd_req)

    bases = []
    if SOURCE == "drive":
        bases.append(Path(DRIVE_PROJECT_PATH))
    bases.append(Path(CLONE_DIR))

    for base in bases:
        if not base.is_dir():
            continue
        direct = base / "requirements.txt"
        if direct.is_file():
            return str(base), str(direct)
        nested = next(base.rglob("requirements.txt"), None)
        if nested is not None:
            return str(nested.parent), str(nested)

    u = REPO_URL.strip().rstrip("/")
    if u.endswith(".git"):
        u = u[:-4]
    m = re.match(r"https?://github\.com/([^/]+)/([^/]+)/?$", u)
    if m:
        owner, repo = m.group(1), m.group(2)
        for branch in ("main", "master"):
            raw = f"https://raw.githubusercontent.com/{owner}/{repo}/{branch}/requirements.txt"
            tmp = "/tmp/requirements_from_github.txt"
            try:
                urllib.request.urlretrieve(raw, tmp)
                if os.path.getsize(tmp) > 80:
                    work = CLONE_DIR if Path(CLONE_DIR).is_dir() else "/content"
                    return work, tmp
            except OSError:
                continue

    return "", ""


root, req_path = find_requirements_file()
if not req_path:
    pinned = """accelerate>=0.33.0
jsonschema>=4.23.0
Pillow>=10.0.0
torch>=2.4.0
transformers>=4.48.0
"""
    req_path = "/tmp/requirements_fallback.txt"
    Path(req_path).write_text(pinned, encoding="utf-8")
    root = CLONE_DIR if Path(CLONE_DIR).is_dir() else os.getcwd()
    print(
        "NOTE: requirements.txt not in clone/GitHub — installing built-in fallback deps. "
        "Push requirements.txt from your PC: git add requirements.txt && git push"
    )

if root and Path(root).is_dir():
    os.chdir(root)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", req_path],
    check=True,
)


NOTE: requirements.txt not in clone/GitHub — installing built-in fallback deps. Push requirements.txt from your PC: git add requirements.txt && git push


CompletedProcess(args=['/usr/bin/python3', '-m', 'pip', 'install', '-q', '-r', '/tmp/requirements_fallback.txt'], returncode=0)

## 5) Optional — Hugging Face login

Uncomment if the model requires authentication.

In [ ]:
# from huggingface_hub import login
# login()

## 6) Run birth-certificate extraction

Finds the BC image folder under this repo (same logic as `find_birth_certificate_image_dir` in `case_study_parser.common`), then runs extract. Writes **`outputs/predictions/`** and **`outputs/raw_model_outputs/`**.

If you see **cannot find case_study_parser**, your GitHub repo does not include the Python package yet — **`git push`** `case_study_parser/`, `prompts/`, `schemas/` from your PC.

If it cannot find **birth certificate images**, push **`data/`** too, then **Runtime → Restart session** and re-run from step 2.

In [45]:
import os
import subprocess
import sys
from pathlib import Path


def resolve_repo_root() -> Path:
    """Find repo root (directory containing case_study_parser/)."""
    candidates = [
        Path.cwd(),
        Path(CLONE_DIR),
        Path(DRIVE_PROJECT_PATH) if SOURCE == "drive" else None,
        Path("/content/Case-Study-Parsing-"),
    ]
    for base in candidates:
        if base is None:
            continue
        pkg = base / "case_study_parser"
        if pkg.is_dir() and (pkg / "__init__.py").is_file():
            os.chdir(base)
            return base.resolve()

    content_root = Path("/content")
    if content_root.is_dir():
        try:
            for marker in content_root.rglob("case_study_parser/__init__.py"):
                base = marker.parent.parent
                os.chdir(base)
                return base.resolve()
        except OSError:
            pass

    hint_lines = []
    cp = Path(CLONE_DIR)
    if cp.is_dir():
        try:
            names = sorted(p.name for p in cp.iterdir())
            hint_lines.append(f"Contents of {cp}: {names}")
        except OSError as e:
            hint_lines.append(f"Could not list {cp}: {e}")
    else:
        hint_lines.append(f"Clone path does not exist: {cp} (re-run section 2).")

    raise FileNotFoundError(
        "Cannot find case_study_parser/ (with __init__.py). "
        "The GitHub clone is often missing code if you only pushed README.\n\n"
        "On your PC, from the project folder, run:\n"
        "  git add case_study_parser prompts schemas data requirements.txt\n"
        "  git commit -m \"Add pipeline code and data\"\n"
        "  git push\n\n"
        + ("\n".join(hint_lines))
    )


repo_root = resolve_repo_root()
if str(repo_root) not in sys.path:
    sys.path.insert(0, str(repo_root))

from case_study_parser.common import find_birth_certificate_image_dir

root = Path.cwd()
bc_dir = find_birth_certificate_image_dir(root)
if bc_dir is None:
    top = list(root.iterdir()) if root.is_dir() else []
    raise FileNotFoundError(
        "Could not find birth certificate images (searched for "
        "data/raw_images/DataSet/Birth Certificate, BC_*.jpeg under repo, "
        "and folders named Birth Certificate).\n"
        f"Repo root has: {[p.name for p in top]}\n"
        "Fix: git add data && git push"
    )

rel = bc_dir.relative_to(root).as_posix()
print("Using birth_certificate=", rel)

subprocess.run(
    [
        sys.executable,
        "-m",
        "case_study_parser.extract",
        "--typed-input-dir",
        f"birth_certificate={rel}",
        "--output-dir",
        "outputs",
        "--model-name",
        "Qwen/Qwen2.5-VL-7B-Instruct",
        "--typed-model",
        "birth_certificate=Qwen/Qwen2.5-VL-7B-Instruct",
    ],
    check=True,
    cwd=str(repo_root),
)


FileNotFoundError: Cannot find case_study_parser/ (with __init__.py). The GitHub clone is often missing code if you only pushed README.

On your PC, from the project folder, run:
  git add case_study_parser prompts schemas data requirements.txt
  git commit -m "Add pipeline code and data"
  git push

Contents of /content/Case-Study-Parsing-: ['.git', 'README.md']

## 7) Zip outputs (download on Colab)

Download **`outputs.zip`** before the runtime disconnects.

In [ ]:
!zip -r /content/outputs.zip outputs/predictions outputs/raw_model_outputs
try:
    from google.colab import files
    files.download("/content/outputs.zip")
except ImportError:
    print("Not on Colab: outputs are in ./outputs/")
    print("Or open /content/outputs.zip if using a remote runtime.")